# **PEFT / LoRA**

### This is the exact technique used to fine-tune:

LLaMA 2 / LLaMA 3

Mistral

Falcon

GPT-Neo / GPT-J

T5 XXL

StableLM

## **Install PEFT**

In [ ]:
!pip install peft transformers accelerate datasets sentencepiece -q

## **Imports**

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType

## **Load Model + Tokenizer**

In [ ]:
model_name = "google/flan-t5-small"
# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

## **Apply LoRA to model**

In [ ]:
lora_config = LoraConfig(
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0.05,
    task_type = TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 688,128 || all params: 77,649,280 || trainable%: 0.8862


## **Load dataset**

In [ ]:
dataset = load_dataset("cnn_dailymail", "3.0.0")

## **Preprocess function**

In [ ]:
def preprocess(batch):
    inputs = ["summarize:"+ doc for doc in batch["article"]]
    targets = batch["highlights"]

    inputs_enc = tokenizer(
        inputs, max_length = 512, truncation = True, padding = "max_length"
    )

    targets_enc = tokenizer(
        targets, max_length = 128, truncation = True, padding = "max_length"
    )

    inputs_enc["labels"] = targets_enc["input_ids"]
    return inputs_enc



## **Tokenize dataset**

In [ ]:
tokenized_train = dataset["train"].map(preprocess, batched=True)
tokenized_val = dataset["validation"].map(preprocess, batched=True)

Map:   0%|          | 0/287113 [00:00<?, ? examples/s]

Map:   0%|          | 0/13368 [00:00<?, ? examples/s]

## **Training arguments**

In [ ]:
training_args = TrainingArguments(
    output_dir = "t5-lora-cnn",
    per_device_train_batch_size = 4,
    per_device_eval_batch_size = 4,
    learning_rate = 1e-4,
    num_train_epochs = 1,
    logging_steps = 100,
    eval_strategy = "steps",
    eval_steps = 200,
    save_steps = 200,
    fp16 = False
)

## **Trainer**

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_val
)

## **Start training**

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: parash-chinnu79 (parash-chinnu79-sri-venkateswara-college-of-engineering) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

## **Save model**

In [ ]:
trainer.save_model("t5-lora-finetuned-cnn")
tokenizer.save_pretrained("t5-lora-finetuned-cnn")

## **set model to evaluation mode**

In [ ]:
model.eval()

## **Predictions**

In [ ]:
def summarize(text):
  inputs = tokenizer("summarize: " + text, return_tensors='pt', truncation=True, padding=True)

  inputs = inputs.to(model.device) # Move inputs to the same device as the model

  outputs = model.generate(**inputs, max_new_tokens=200, min_length=70, num_beams=4, length_penalty = 1.2, early_stopping=True)
  summary = tokenizer.decode(outputs[0], skip_special_tokens = True)

  return summary

print(summarize(
    """
    Artificial Intelligence is transforming industries worldwide.
    Companies are adopting automation, predictive analytics, and advanced
    AI-driven systems to improve efficiency, reduce costs, and innovate new services.
    """
))

## **Gradio UI**

In [ ]:
# ------------------------------------------------------------
# 🧠 Install dependencies
# ------------------------------------------------------------
!pip install gradio transformers peft sentencepiece accelerate -q

# ------------------------------------------------------------
# 🧠 Imports
# ------------------------------------------------------------
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

# ------------------------------------------------------------
# 🧠 Load Base Model + LoRA Weights
# ------------------------------------------------------------
base_model_name = "google/flan-t5-small"
lora_model_path = "t5-lora-finetuned-cnn"    # your saved LoRA folder

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)

# Load LoRA weights
model = PeftModel.from_pretrained(base_model, lora_model_path)
model.eval()  # inference mode


# ------------------------------------------------------------
# 🧠 Summarization Function
# ------------------------------------------------------------
def generate_summary(text, max_tokens, min_tokens, beams, length_penalty):
    if not text.strip():
        return "⚠️ Please enter some text to summarize."

    # Add the T5 instruction
    prompt = "summarize: " + text

    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True)

    # Generate summary
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        min_length=min_tokens,
        num_beams=beams,
        length_penalty=length_penalty,
        early_stopping=True
    )

    # Decode tokens → text summary
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return summary


# ------------------------------------------------------------
# 🧠 Gradio UI Layout
# ------------------------------------------------------------
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("<h1 style='text-align:center;'>📘 LoRA-Tuned T5 Summarizer</h1>")
    gr.Markdown("<p style='text-align:center;'>Enter any article or long text and get an AI-generated summary.</p>")

    with gr.Row():
        text_input = gr.Textbox(
            lines=10,
            label="Enter the Article / Text",
            placeholder="Paste a long text here for summarization..."
        )

    with gr.Accordion("⚙️ Advanced Settings (optional)", open=False):
        max_tokens = gr.Slider(50, 300, value=200, label="Max New Tokens")
        min_tokens = gr.Slider(10, 150, value=50, label="Min Summary Length")
        beams = gr.Slider(1, 6, value=4, step=1, label="Beam Search Width (num_beams)")
        length_penalty = gr.Slider(0.1, 2.0, value=1.2, label="Length Penalty")

    summarize_btn = gr.Button("🔍 Generate Summary")
    output_box = gr.Textbox(label="Summary Output", lines=8)

    summarize_btn.click(
        fn=generate_summary,
        inputs=[text_input, max_tokens, min_tokens, beams, length_penalty],
        outputs=output_box
    )

# ------------------------------------------------------------
# 🧠 Launch the App
# ------------------------------------------------------------
demo.launch()
